In [ ]:
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://sandbox.figuard.io",
)

# Orchestrator creates the parent fleet budget
fleet = client.create_budget(
    user_id="orchestrator",
    total_limit=1000.00,
    currency="USD",
    expires_in="24h",
    intent_context="multi-agent research and booking",
)

print(f"✓ Fleet budget created: {fleet.id}")
print(f"  Total limit: ${fleet.total_limit}")

# Issue scoped delegation tokens per sub-agent
research_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet.session_token,
    label="research_agent",
    caps=[{"category": "api", "limit": 200.00}],
    expires_in="4h",
)

booking_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet.session_token,
    label="booking_agent",
    caps=[{"category": "flight", "limit": 600.00}],
    expires_in="4h",
)

print(f"✓ research_agent cap: $200 (api)")
print(f"✓ booking_agent  cap: $600 (flight)")

In [ ]:
# research_agent spends against its delegation token
auth = client.authorize(
    session_token=research_token.session_token,
    agent_id="research_agent",
    action_type="EXTERNAL_CALL",
    description="Serper API: flight route research",
    requested_quantity=15.00,
    claimed_category="api",
    idempotency_key="research-001",
)
print(f"Research API call: {auth.decision} — ${auth.approved_quantity}")
client.confirm_event(auth.event_id, confirmed_quantity=15.00)
print("✓ Confirmed.")

# booking_agent spends against its delegation token
auth2 = client.authorize(
    session_token=booking_token.session_token,
    agent_id="booking_agent",
    action_type="PURCHASE",
    description="Delta JFK→LAX roundtrip",
    requested_quantity=450.00,
    claimed_category="flight",
    idempotency_key="booking-001",
)
print(f"Flight booking:    {auth2.decision} — ${auth2.approved_quantity}")
client.confirm_event(auth2.event_id, confirmed_quantity=450.00)
print("✓ Confirmed.")

# booking_agent tries to exceed its cap ($600 - $450 = $150 left)
auth3 = client.authorize(
    session_token=booking_token.session_token,
    agent_id="booking_agent",
    action_type="PURCHASE",
    description="Business class upgrade",
    requested_quantity=400.00,
    claimed_category="flight",
    idempotency_key="booking-002",
)
print(f"Upgrade attempt:   {auth3.decision} — {auth3.denial_reason}")
print("Delegation cap enforced. Fleet total unaffected.")

print(f"\n→ See the spend tree: https://sandbox.figuard.io/ui")